# Optimización Bajo Incertidumbre (IIND 4125)

## Proyecto 3: Decisiones Secuenciales — MDP, Políticas y Aprendizaje

**Formato:** reporte ejecutivo + notebook reproducible + presentación en clase

## Motivación

En los Proyectos 1 y 2, cada grupo formuló un problema de optimización bajo incertidumbre y lo resolvió tomando todas las decisiones de una vez (primera etapa) frente a escenarios de segunda etapa. Esa formulación "aquí y ahora" es poderosa, pero ignora que en la mayoría de los sistemas reales, las decisiones se toman a lo largo del tiempo, observando nueva información entre cada decisión.

Un operador logístico no planea todas sus rutas del mes el día 1 — cada mañana observa la demanda real del día anterior, el estado de su flota, y decide qué hacer *hoy*. Un gestor de energía no fija su estrategia de almacenamiento para toda la semana — cada hora observa precios, generación solar y demanda, y decide cuánto cargar o descargar *ahora*. Un hospital no asigna todas sus camas el lunes — cada turno observa ingresos, altas y urgencias, y redistribuye recursos *en el momento*.

Este proyecto les pide reformular su problema como uno de decisiones secuenciales, modelándolo como un Proceso de Decisión de Markov (MDP), e implementando y comparando distintas familias de políticas para resolverlo.

El resultado será una cadena computacional completa: **ambiente de simulación → políticas → entrenamiento → evaluación → comparación**.


## 1. Punto de Partida: Reformulación Secuencial

Cada grupo retoma el contexto de su problema de los proyectos anteriores (o una variante) y lo reformula como un problema de decisiones secuenciales. No se cambia de dominio de aplicación.

### 1.1 Reformulación como MDP (Powell)

Deben declarar explícitamente los cinco elementos del marco unificado de Powell:

| Elemento | Símbolo | Descripción | Ejemplo (inventario) |
|---|---|---|---|
| **Estado** | $S_t$ | Información física + de conocimiento en $t$ | Nivel de inventario $R_t$ + régimen de demanda |
| **Decisión** | $x_t$ | Variables que controlan en cada periodo | Cantidad a ordenar $q_t$ |
| **Información exógena** | $W_{t+1}$ | Nueva información incierta que llega | Demanda realizada $D_{t+1}$, precio spot |
| **Función de transición** | $S^M$ | $S_{t+1} = S^M(S_t, x_t, W_{t+1})$ | $R_{t+1} = R_t + q_t - \min(D_{t+1}, R_t + q_t)$ |
| **Función de costo/beneficio** | $C_t$ | Contribución por periodo | $C_t = c \cdot q_t + h \cdot [R_t + q_t - D_t]^+ + p \cdot [D_t - R_t - q_t]^+$ |

**Requisito de continuidad:** Los generadores de incertidumbre ($W$ y $\hat{W}$, Mediocristan y Extremistan) deben ser coherentes con los de los Proyectos anteriores, pero ahora generan secuencias temporales $W_1, W_2, \ldots, W_T$ (no solo conjuntos de escenarios estáticos).


## 2. Ambiente de Simulación

Implementen un ambiente de simulación modular con los siguientes componentes:

### 2.1 Generador de Variables Aleatorias

Función `generate_W(T, seed, mode)` que produce una secuencia temporal de información exógena para un episodio de $T$ periodos.

- **`mode='mediocristan'`**: ruido moderado, distribuciones clásicas (coherente con Proyecto 1).
- **`mode='extremistan'`**: colas pesadas, cambios de régimen, shocks (coherente con Proyecto 1).
- La semilla (`seed`) debe garantizar reproducibilidad.

### 2.2 Función de Transición

Función `transition(S_t, x_t, W_{t+1})` → $S_{t+1}$ que actualiza el estado del sistema.

- Debe respetar las restricciones físicas (capacidad, no-negatividad, conservación de flujo, etc.).
- Debe ser determinista dado $(S_t, x_t, W_{t+1})$ — toda la estocasticidad vive en $W$.

### 2.3 Función de Costo

Función `period_cost(S_t, x_t, W_{t+1})` → $C_t$ que calcula la contribución (costo o beneficio) del periodo.

- Debe capturar los trade-offs relevantes: costos operativos, penalizaciones por déficit/exceso, costos de oportunidad, etc.

### 2.4 Simulador de Episodios

Función `run_episode(policy_fn, W, params)` que ejecuta un episodio completo:

```
Para t = 0, 1, ..., T-1:
    1. Observar estado S_t
    2. Tomar decisión x_t = policy_fn(S_t, params)
    3. Observar información exógena W_{t+1}
    4. Transitar S_{t+1} = transition(S_t, x_t, W_{t+1})
    5. Registrar costo C_t = period_cost(S_t, x_t, W_{t+1})
Retornar: historial de estados, decisiones, costos
```


## 3. Políticas: Implementación y Comparación

Implementen políticas de familias distintas de la taxonomía de Powell. Como mínimo obligatorio, deben incluir:

- **Una política de tipo DLA** (Direct Lookahead Approximation)
- **Una política de tipo VFA** (Value Function Approximation)

Se recomienda complementar con políticas más simples como *baseline*.

### 3.1 Baselines (recomendados, no obligatorios)

#### PFA — Policy Function Approximation
Reglas parametrizadas que mapean estado a acción directamente:
- Regla de umbral (e.g., "ordenar si inventario < $s$, hasta nivel $S$")
- Regla lineal (señales con pesos calibrables)
- Heurística del dominio

#### CFA — Cost Function Approximation
Resolver un modelo de optimización proxy en cada periodo:
- LP/MIP con costos o restricciones modificadas (buffers, penalizaciones)
- Parámetros tuneables: nivel de conservadurismo, safety stock, penalización por déficit

### 3.2 VFA — Value Function Approximation (obligatorio)

Implementen al menos una de las siguientes variantes:

| Variante | Requiere modelo de transición | Método |
|---|---|---|
| **Value Iteration (VI)** | Sí | Bellman exacto sobre estados discretizados |
| **Policy Iteration (PI)** | Sí | Evaluación + mejora iterativa |
| **Q-Learning** | No (model-free) | TD off-policy, actualiza $Q(s,a)$ con $\max_{a'} Q(s', a')$ |
| **SARSA** | No (model-free) | TD on-policy, actualiza $Q(s,a)$ con $Q(s', a')$ donde $a'$ es la acción tomada |

**Entregable VFA:**
- Curva de convergencia (error de Bellman o retorno promedio vs iteración/episodio).
- Si usaron discretización: explicar y justificar la resolución elegida.
- Visualización de la política aprendida (e.g., heatmap de acciones por estado).

### 3.3 DLA — Direct Lookahead Approximation (obligatorio)

Implementen al menos una de las siguientes variantes:

| Variante | Descripción |
|---|---|
| **Rolling Horizon determinista** | Resolver LP/MIP sobre horizonte $H$ con pronóstico puntual; ejecutar solo $x_t$ |
| **Rolling Horizon estocástico** | Igual, pero con escenarios sobre el horizonte $H$ |
| **MCTS (Monte Carlo Tree Search)** | Exploración selectiva de un árbol de acciones/estados con rollouts |

**Entregable DLA:**
- Análisis de sensibilidad al horizonte $H$ (o profundidad/iteraciones en MCTS).
- Comparación de calidad de pronóstico vs desempeño de la política.

### 3.4 Calibración y Selección de Políticas

El proceso de selección tiene dos niveles:

#### Nivel 1: Selección intra-familia
Para cada familia de políticas, corran episodios de entrenamiento/calibración para elegir la mejor variante o parametrización:

- **PFA/CFA:** grid search o búsqueda aleatoria sobre parámetros $\theta$.
- **VFA:** comparar VI vs Q-Learning, o diferentes tasas de aprendizaje / discretizaciones.
- **DLA:** comparar horizontes $H$, número de escenarios, o iteraciones de MCTS.

#### Nivel 2: Torneo inter-familia
Con el mejor representante de cada familia, corran un torneo final de $N \geq 50$ episodios de evaluación (semillas distintas a las de entrenamiento) y comparen:

| Métrica | Descripción |
|---|---|
| Costo promedio | $\bar{C} = \frac{1}{N} \sum_{n=1}^{N} \sum_{t=0}^{T-1} C_t^{(n)}$ |
| Desviación estándar | Variabilidad entre episodios |
| CVaR$_{90}$ | Costo esperado en el 10% peor de los episodios |
| Tasa de violación | Frecuencia de violación de restricciones (déficit, capacidad, etc.) |



## 4. Visualizaciones

Las visualizaciones son obligatorias y deben permitir entender *qué hace* cada política, no solo *cuánto cuesta*. Como mínimo:

### 4.1 Anatomía de decisiones
Para un episodio representativo de cada política, graficar en el tiempo:
- Estado $S_t$ (e.g., nivel de inventario, flujo en red)
- Decisión $x_t$ (e.g., cantidad ordenada, flujo asignado)
- Información exógena $W_t$ (e.g., demanda realizada, precio)
- Costo acumulado $\sum_{\tau=0}^{t} C_\tau$

### 4.2 Comparación estadística
- **Boxplots** de costo total por política.
- **Tabla resumen** con media, std, CVaR, tasa de violación.

### 4.3 Convergencia y aprendizaje
- **VFA:** curva de convergencia (Bellman residual o retorno por episodio de entrenamiento).
- **DLA:** sensibilidad al horizonte $H$.
- **PFA/CFA:** landscape de búsqueda de parámetros.

## 5. Análisis Comparativo y Recomendaciones

### 5.1 Comparación Mediocristan vs Extremistan (opcional)

Repitan el torneo inter-familia bajo ambos mundos. Analicen:

1. ¿Qué política es más robusta (menor degradación al pasar de Mediocristan a Extremistan)?
2. ¿Qué política tiene mejor desempeño en cola (CVaR más bajo en Extremistan)?
3. ¿Hay alguna política que sea la mejor en un mundo pero la peor en el otro?

### 5.2 Recomendaciones para Stakeholders

Redacten una sección de recomendaciones (máx. 1 página) dirigida a un tomador de decisiones no técnico:
- ¿Qué política recomiendan y por qué?
- ¿Cuánto vale la sofisticación adicional (VFA/DLA) respecto al baseline (PFA)?
- ¿Qué tan sensible es la recomendación al tipo de incertidumbre?

## 6. Lecturas

Todos los grupos deben leer:

- Powell, W. B. (2019). A unified framework for stochastic optimization. European journal of operational research, 275(3), 795-821.

Y **un** artículo relacionado con su dominio de aplicación y alguna(s) de las técnicas que se discuten en el artículo obligatorio.

## 7. Entregables

### A) Reporte ejecutivo (máx. 8 páginas) + presentación en clase (< 8 min)

1. **Recordatorio del problema** (breve, referenciando Proyectos 1 y 2).
2. **Formulación MDP** completa: tabla de los 5 elementos de Powell con notación clara.
3. **Descripción del ambiente** de simulación: generador, transición, costo.
4. **Políticas implementadas:** descripción conceptual y algorítmica de cada una.
5. **Calibración intra-familia:** cómo eligieron la mejor variante de cada familia.
6. **Torneo inter-familia:** tabla y gráficos comparativos (Mediocristan y Extremistan).
7. **Análisis de riesgo:** comparación de robustez y desempeño en colas.
8. **Síntesis de lecturas:** resumen de Powell/Sutton + dos lecturas adicionales (máx. 1 página en total).
9. **Recomendaciones para stakeholders.**

### B) Notebook reproducible

El notebook debe:

- Definir el ambiente de simulación (generador, transición, costo) de forma modular.
- Implementar al menos una política VFA y una DLA (más baselines recomendados).
- Ejecutar calibración intra-familia y torneo inter-familia.
- Producir todas las tablas, figuras y visualizaciones del reporte.
- Correr de principio a fin sin errores (probado por ustedes antes de entregar).
- Usar semillas fijas para reproducibilidad.

### C) Mini-bibliografía comentada

- Síntesis de la lectura obligatoria (Powell).
- Síntesis del artículo elegido.

## 8. Notas Prácticas y Restricciones

- **Modularidad.** Los cinco componentes (Oracle, Forecast, Policy, Simulator, Analytics) deben estar desacoplados. Deben poder agregar una nueva política sin modificar el simulador, y viceversa.
- **Correctitud antes que complejidad.** Un MDP pequeño con 2 políticas bien implementadas y comparadas vale más que uno grande con bugs silenciosos. Verifiquen contra baselines y contra intuición del dominio.
- **Reproducibilidad.** Semillas, dependencias, instrucciones claras. Si no corre, no se puede evaluar.
- **Las visualizaciones deben contar una historia.** No basta con boxplots — muestren *qué* decide cada política y *por qué* una es mejor que otra.
- **El modelo no tiene que ser perfecto.** La función de transición puede ser una simplificación razonable de la realidad. Lo importante es que sea coherente, esté bien documentada, y permita comparar políticas de forma justa.

## Mapa de Competencias: Proyecto 1 → Proyecto 2 → Proyecto 3

```
Proyecto 1                    Proyecto 2                    Proyecto 3
──────────                    ──────────                    ──────────
Formulación del problema  →   Reformulación two-stage   →   Reformulación MDP secuencial
Escenarios fijos (pocos)  →   SAA con N variable        →   Episodios de simulación (T pasos)
Resolver DE monolítico    →   Descomponer (Benders)     →   Políticas (PFA, CFA, VFA, DLA)
x_det, x_sto, x_risk     →   + CVaR risk-averse        →   Política robusta vs risk-neutral
Backtesting simple        →   VSS, EVPI, CVaR           →   Torneo de políticas + stress test
Mediocristan vs Extremistan   Mismo contraste, mayor rigor   Mismo contraste, decisiones en el tiempo
```

---

## Taxonomía de Políticas: Guía Rápida

```
┌─────────────────────────────────────────────────────────────────┐
│                    POLÍTICAS (Powell)                            │
│                                                                 │
│  PFA                    CFA                                     │
│  "Regla directa"        "Optimización con costos proxy"         │
│  x = f(S; θ)           x = argmin C_proxy(x; θ)                │
│  Ejemplos:              Ejemplos:                               │
│  · (s,S) policy         · LP con safety stock                   │
│  · Umbral de precio     · MIP con penalización ajustable        │
│  · Red neuronal (PG)    · Robust counterpart parametrizado      │
│                                                                 │
│  VFA                    DLA                                     │
│  "Valor del futuro"     "Mirar hacia adelante"                  │
│  x = argmin C(x) +     x₀ de: min Σ C_t sobre                  │
│      V̄(S')              horizonte H con escenarios              │
│  Ejemplos:              Ejemplos:                               │
│  · Value Iteration      · Rolling horizon LP/MIP                │
│  · Q-Learning           · Stochastic programming multietapa     │
│  · SARSA                · MCTS                                  │
│  · ADP / DQN            · MPC                                   │
└─────────────────────────────────────────────────────────────────┘
```
